<a href="https://colab.research.google.com/github/mayajdias/ds2002-fa26/blob/main/notebooks/01-foundations/2026_10_09_public_api_to_clean_table_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_problem = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError) as e:
            last_problem = repr(e)           # network flake: worth retrying
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                # 400/404 etc. are our bug, so fail immediately
                raise RuntimeError(f'HTTP {r.status_code}: {r.text[:200]}')
            last_problem = f'HTTP {r.status_code}'
        if attempt < tries - 1:
            time.sleep(2 ** attempt)         # 1s, 2s, 4s...
    raise RuntimeError(f'get_weather failed after {tries} tries: {last_problem}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
from datetime import datetime

DATES = ['2026-09-05', '2026-10-03']
used_fallback = False
raw = None

try:
    raw = get_weather(DATES[0], DATES[1])        # one request spanning both dates
    pulled_at = datetime.now()
    raw = raw[raw['time'].isin(DATES)]           # keep just the two dates
except Exception as e:
    print(f'API call failed, using fallback: {e}')
    used_fallback = True
    pulled_at = datetime.now()

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = raw.copy()
    wx['source'] = 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].reset_index(drop=True)
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']

if diff == 0:
    print('Both dates had the same rainfall.')
else:
    print(f"{wet['date']:%B %d, %Y} was wetter than {dry['date']:%B %d, %Y}: "
          f"{wet['precip_mm']:.1f} mm of rain versus {dry['precip_mm']:.1f} mm, "
          f"a difference of {diff:.1f} mm (source: {wet['source']}).")

October 03, 2026 was wetter than September 05, 2026: 14.7 mm of rain versus 2.5 mm, a difference of 12.2 mm (source: open-meteo).


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print('Endpoint:   ', BASE)
print('Coordinates: 38.03, -78.51')
print('Date range: ', DATES[0], 'to', DATES[1])
print('Pulled at:  ', pulled_at.strftime('%Y-%m-%d %H:%M:%S'))
print('Data type:  ', 'FALLBACK SAMPLE' if used_fallback else 'real data from Open-Meteo')

Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range:  2026-09-05 to 2026-10-03
Pulled at:   2026-10-09 03:40:14
Data type:   real data from Open-Meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure, the merge onto 200 days of sales would run without any error. A left join would keep all the sales rows but fill every weather column with NaN, and an inner join would silently drop every row. Plotting rain against revenue would then produce either an empty chart or a chart with no points (most plotting libraries skip NaN quietly), or, after a careless fillna(0), a chart implying it never rained and that rain has no relationship to revenue. Nobody would catch it because nothing crashed, the code looks fine, and “weather had no effect” is a plausible-sounding finding. The failure moves from the console into the conclusions. Raising an exception, or at least asserting on row count and NaNs like Part 5, forces the problem to surface at the source.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
CITIES = {'Charlottesville': (38.03, -78.51), 'Orlando': (28.54, -81.38)}
frames = []

for city, (lat, lon) in CITIES.items():
    df = get_weather(DATES[0], DATES[1], lat=lat, lon=lon)
    df = df[df['time'].isin(DATES)].rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    df['date'] = pd.to_datetime(df['date'])
    df['city'] = city
    df['source'] = 'open-meteo'
    frames.append(df[['city', 'date', 'temp_max', 'precip_mm', 'source']])

both = pd.concat(frames, ignore_index=True)
both

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
